In [1]:
# This is necessary to recognize the modules
import os
import sys
import warnings

warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)

In [2]:
from core.services.timescale_client import TimescaleClient

# Create test connection to the new backtesting database
ts_client = TimescaleClient(
    host="localhost",
    port=5433,  # Using the new port we specified
    user="backtest_user",
    password="backtest_password",
    database="backtest_db"
)

# Test connection
await ts_client.connect()


In [3]:
import datetime
from core.backtesting.optimizer import StrategyOptimizer

In [4]:

from research_notebooks.elitesmugplug.elitesmugplug_config_gen_simple import SmugPlug3LiteConfigGenerator

# Define the date range for the optimization
start_date = datetime.datetime(2025, 1, 1)
end_date = datetime.datetime(2025, 2, 1)

# Create the configuration generator
config_generator = SmugPlug3LiteConfigGenerator(start_date=start_date, end_date=end_date)


In [5]:
if 'optimizer' in globals():
    await optimizer.close()

# Clean up any stale connections without deleting data
import sqlite3

def unlock_database():
    try:
        # Force close any existing connections
        conn = sqlite3.connect('optuna.db')
        conn.execute("PRAGMA busy_timeout = 5000")
        conn.execute("PRAGMA journal_mode = DELETE")
        conn.close()
        print("Database unlocked successfully")
    except Exception as e:
        print(f"Error during unlock: {str(e)}")

unlock_database()

Database unlocked successfully


In [6]:
# First, close any existing connections
if 'optimizer' in globals():
    await optimizer.close()

import optuna
from optuna.storages import RDBStorage

# Define TimescaleDB configuration
timescale_config = {
    "host": "localhost",
    "port": 5433,
    "user": "backtest_user",
    "password": "backtest_password",
    "database": "backtest_db"
}

# First, close any existing connections
if 'optimizer' in globals():
    await optimizer.close()

# Create the optimizer with correct parameters
optimizer = StrategyOptimizer(
    root_path=root_path,
    storage_name="sqlite:///optuna.db",  # Using SQLite for now
    load_cached_data=False,
    resolution="1m"
)


In [7]:

# Launch the Optuna dashboard
optimizer.launch_optuna_dashboard()

# Run the optimization
await optimizer.optimize(
    study_name=f"elitesmugplug2",
    config_generator=config_generator,
    n_trials=10000,
)
optimizer.get_study_best_params(f"elitesmugplug2")

I0000 00:00:1740933216.926578 9954741 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers
[I 2025-03-02 11:33:37,105] A new study created in RDB with name: elitesmugplug2
2025-03-02 11:33:38,168 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16a0aa0e0>
2025-03-02 11:33:38,172 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x1571101c0>, 405862.007997)])']
connector: <aiohttp.connector.TCPConnector object at 0x16a0a84c0>
Listening on http://127.0.0.1:8080/
Hit Ctrl-C to quit.

127.0.0.1 - - [02/Mar/2025 11:33:46] "GET /api/studies/9?after=1368 HTTP/1.1" 404 36
2025-03-02 11:33:47,297 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16a0abf40>
2025-03-02 11:33:47,307 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x16a062

CancelledError: 

127.0.0.1 - - [02/Mar/2025 13:55:07] "GET /api/studies/3/param_importances HTTP/1.1" 200 1147
127.0.0.1 - - [02/Mar/2025 13:55:17] "GET /api/studies/3?after=446 HTTP/1.1" 200 45083
127.0.0.1 - - [02/Mar/2025 13:55:28] "GET /api/studies/3?after=446 HTTP/1.1" 200 45083


In [ ]:
import datetime
from core.backtesting.optimizer import StrategyOptimizer
from research_notebooks.elitesmugplug.elitesmugplug_config_gen_simple import SmugPlug3LiteConfigGenerator

# Define the date range for the optimization
start_date = datetime.datetime(2025, 1, 1)
end_date = datetime.datetime(2025, 2, 1)

# Database configuration for Docker container
timescale_config = {
    "host": "localhost",  # or your Docker host IP
    "port": 5432,        # exposed port from docker-compose
    "user": "admin",     # from docker-compose
    "password": "admin", # from docker-compose
    "database": "timescaledb"
}

# Create the configuration generator
config_generator = SmugPlug3LiteConfigGenerator(
    start_date=start_date, 
    end_date=end_date,
    timescale_config=timescale_config
)

# Create the optimizer with database connection
optimizer = StrategyOptimizer(
    root_path=root_path,
    db_host=timescale_config["host"],
    db_port=timescale_config["port"],
    db_user=timescale_config["user"],
    db_pass=timescale_config["password"]
)

# Launch the Optuna dashboard
optimizer.launch_optuna_dashboard()

# Run the optimization
await optimizer.optimize(
    study_name=f"smugplug_optimization",
    config_generator=config_generator,
    n_trials=100,
)